In [2]:
%pip install --user pandas numpy matplotlib seaborn sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --user pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys, site
print("KERNEL:", sys.executable)
print("USER SITE:", site.getusersitepackages())
print("PATHS:", [p for p in sys.path if 'site-packages' in p])

KERNEL: C:\Program Files\Python313\python.exe
USER SITE: C:\Users\ManeV1\AppData\Roaming\Python\Python313\site-packages
PATHS: ['C:\\Users\\ManeV1\\AppData\\Roaming\\Python\\Python313\\site-packages', 'C:\\Program Files\\Python313\\Lib\\site-packages']


## 1. Load

In [5]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv(r'C:\IT_service_desk_data_analysis\it_service_desk_tickets_2024_2026.csv', dtype=str)

print(df.shape)
df.head()

(44250, 29)


,Call Date,Call Number,Caller Name,Store Number,Store Name,Region,Brief Description (Details),Category,Subcategory,Status,Priority,Completion Date,Target Date,Supplier,Completed,Operator Group,Operator,Contact Channel,First Call Resolution,KnowledgeBaseAvailable,Time spent for First line,Self Service Eligible,Automation Suitability,Reopened Count,Customer Satisfaction Score,Record Version,Source System,Extract Flag,Legacy Ref ID
0,06/11/2024 16:44,C2411-22359,1091STUK OSWESTRY Store Employee,1091,OSWESTRY,Scotland & NI,store wifi dropping,UK - Store Network,WiFi Connection,Closed,P3,26/11/2024 16:49,08/11/2024 16:44,UK - Vertex,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,No,00:28:25,No,Low,0,3,1,TOPdesk,Y,LG-878828
1,07/02/2026 07:08,C2602-20950,1203STUK KENDAL Store Employee,1203,KENDAL,Scotland & NI,body cam dock fault,UK - Store Security,BWC Docking Controller,Closed,P3,19/02/2026 16:43,09/02/2026 07:08,NaN,TRUE,UK - SD IT Service Desk,Sofia Almeida,Phone,No,Yes,01:40:25,No,Low,0,3,1,TOPdesk,Y,LG-213624
2,18/05/2026 09:58,C2605-20993,1854STUK CARLISLE Store Employee,1854,CARLISLE,London & South East,password reset request,UK - Store Access,Password Reset - Intranet,Closed,P4,18/05/2026 10:20,23/05/2026 09:58,NaN,TRUE,UK - SD IT Service Desk,Daniel Okafor,Email,Yes,Yes,00:07:38,Yes,High,0,4,1,TOPdesk,Y,LG-669451
3,12/05/2026 12:00,C2605-20337,1014STUK CARLISLE Store Employee,1014,CARLISLE,London & South East,chip and pin not taking payments,UK - Store Till,Chip & Pin,Closed,P3,20/05/2026 09:54,14/05/2026 12:00,NaN,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,Yes,00:27:12,No,Low,0,NaN,1,TOPdesk,Y,LG-358322
4,25/11/2024 09:19,C2411-20650,1560STUK YEOVIL Store Employee,1560,YEOVIL,London & South East,body cam offline,UK - Store Security,BWC Camera Offline,Closed,Priority 3,04/12/2024 19:47,27/11/2024 09:19,NaN,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,No,00:16:43,No,Low,0,5,1,TOPdesk,Y,LG-208444


In [6]:
# First look
df.info()
df.isna().sum().sort_values(ascending=False)
df.duplicated().sum()
df['Subcategory'].nunique(), df['Category'].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 44250 entries, 0 to 44249
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Call Date                    44250 non-null  str  
 1   Call Number                  44250 non-null  str  
 2   Caller Name                  44250 non-null  str  
 3   Store Number                 43584 non-null  str  
 4   Store Name                   43719 non-null  str  
 5   Region                       43804 non-null  str  
 6   Brief Description (Details)  42923 non-null  str  
 7   Category                     44250 non-null  str  
 8   Subcategory                  43133 non-null  str  
 9   Status                       44250 non-null  str  
 10  Priority                     43360 non-null  str  
 11  Completion Date              42915 non-null  str  
 12  Target Date                  44250 non-null  str  
 13  Supplier                     4650 non-null   str  
 14  C

(69, 14)

## 2. Cleaning

In [7]:
raw_rows = len(df)
df.columns = df.columns.str.strip()

### Drop junk columns

In [8]:
df = df.drop(columns=['Record Version', 'Source System', 'Extract Flag', 'Legacy Ref ID'])

### Strip whitespace from all text

In [9]:
text_cols = df.select_dtypes(include='str').columns
for c in text_cols:
    df[c] = df[c].str.strip()
df = df.replace({'': np.nan})

### Duplicates

In [10]:
df = df.drop_duplicates()                              # exact
df = df.drop_duplicates(subset=['Call Number'], keep='first')   # duplicate ticket IDs
print(f"{raw_rows} -> {len(df)} rows")

44250 -> 43001 rows


### Standardise Category

In [11]:
df['Category'] = df['Category'].str.title().str.replace(r'\s+', ' ', regex=True)
df['Category'] = df['Category'].replace({'Uk - Store Hht': 'UK - Store HHT'})
df['Category'] = df['Category'].str.replace('Uk - ', 'UK - ', regex=False)
df['Category'].value_counts()

Category
UK - Store Access        13064
UK - Store Till          10552
UK - Store HHT           10074
UK - Store Backoffice     4108
UK - Store Network        2442
UK - Store Security       1511
UK - Store Telephony      1250
Name: count, dtype: int64

### Standardise Subcategory — the big one, 69 variants collapse to ~43

In [12]:
def clean_sub(s):
    if pd.isna(s):
        return np.nan
    s = s.strip().lower()
    s = s.replace(' and ', ' & ')
    s = re.sub(r'\s*/\s*', '/', s)      # "Server / Base Unit" -> "server/base unit"
    s = re.sub(r'\s*-\s*', ' - ', s)    # normalise dashes
    s = re.sub(r'\s+', ' ', s)
    s = s.replace('reciept', 'receipt')  # typo
    return s.title()

df['Subcategory'] = df['Subcategory'].apply(clean_sub)
print(df['Subcategory'].nunique())
sorted(df['Subcategory'].dropna().unique())

43


['Account Locked',
 'Base Unit',
 'Brother Printer',
 'Bwc Camera Offline',
 'Bwc Docking Controller',
 'Cable/Port Issue',
 'Cash Drawer',
 'Chip & Pin',
 'Chip & Pin Ipc Service',
 'Click & Collect App',
 'Damage',
 'Detagger',
 'End Of Day',
 'Freezing',
 'Handset Not Working',
 'Hht Bluetooth Printer',
 'Hht Charging/Battery Issue',
 'Hht Cradle/Charging Cable',
 'Hht Touchscreen',
 'Leaver Account Removal',
 'Login Issue',
 'Monitor',
 'New Starter Account Setup',
 'No Dial Tone',
 'Offline',
 'Password Reset - Edge',
 'Password Reset - Intranet',
 'Password Reset - Oracle',
 'Performance',
 'Pharmacy Email Error',
 'Receipt Printer',
 'Reports',
 'Rsim App',
 'Scanner',
 'Scanner Not Working',
 'Server/Base Unit',
 'Siso App',
 'Store Network Down',
 'Task Issue',
 'Topdesk - Login Issues',
 'Touch Screen',
 'Wifi',
 'Wifi Connection']

### Priority

In [13]:
df['Priority'] = (df['Priority']
                  .str.replace('Priority ', 'P', regex=False)
                  .str.upper().str.strip())
df['Priority'].value_counts(dropna=False)

Priority
P3     25002
P4     12310
P2      4238
NaN      859
P1       592
Name: count, dtype: int64

### Status, FCR, KB flag 

In [14]:
df['Status'] = df['Status'].str.title()

df['First Call Resolution'] = df['First Call Resolution'].str.strip().str.upper().str[0]
df['First Call Resolution'] = df['First Call Resolution'].map({'Y': 'Yes', 'N': 'No'})

df['KnowledgeBaseAvailable'] = df['KnowledgeBaseAvailable'].str.strip().str.upper().str[0]
df['KnowledgeBaseAvailable'] = df['KnowledgeBaseAvailable'].map({'Y': 'Yes', 'N': 'No'})

### Store fields

In [15]:
df['Store Name'] = df['Store Name'].str.strip().str.upper()
df['Store Number'] = pd.to_numeric(df['Store Number'], errors='coerce').astype('Int64')

# Recover missing store number/name from Caller Name
extracted = df['Caller Name'].str.extract(r'(?P<num>\d+)STUK\s+(?P<name>[A-Z ]+?)\s+Store')
df['Store Number'] = df['Store Number'].fillna(pd.to_numeric(extracted['num'], errors='coerce').astype('Int64'))
df['Store Name'] = df['Store Name'].fillna(extracted['name'].str.strip())

# Region is a store attribute - backfill from other rows for the same store
region_map = df.dropna(subset=['Region']).groupby('Store Number')['Region'].agg(lambda x: x.mode()[0])
df['Region'] = df['Region'].fillna(df['Store Number'].map(region_map))

### Dates — mixed formats

In [16]:
for col in ['Call Date', 'Completion Date', 'Target Date']:
    df[col] = pd.to_datetime(df[col], format='mixed', dayfirst=True, errors='coerce')

df[['Call Date', 'Completion Date', 'Target Date']].isna().sum()

# Sanity check: completion should never precede the call
bad = df['Completion Date'] < df['Call Date']
print(f"{bad.sum()} invalid completion dates")
df.loc[bad, 'Completion Date'] = pd.NaT

0 invalid completion dates


### Time spent → minutes

In [17]:
df['Time spent for First line'] = pd.to_timedelta(df['Time spent for First line'], errors='coerce')
df['handling_minutes'] = df['Time spent for First line'].dt.total_seconds() / 60

# 00:00:00 is a data-entry artefact, not a real zero-effort ticket
df.loc[df['handling_minutes'] == 0, 'handling_minutes'] = np.nan

# Impute from the subcategory median so total-hours calcs aren't understated
df['handling_minutes'] = df['handling_minutes'].fillna(
    df.groupby('Subcategory')['handling_minutes'].transform('median')
)
df['handling_minutes'] = df['handling_minutes'].fillna(df['handling_minutes'].median())

### Numeric + derived fields

In [18]:
df['Reopened Count'] = pd.to_numeric(df['Reopened Count'], errors='coerce').fillna(0).astype(int)
df['Customer Satisfaction Score'] = pd.to_numeric(df['Customer Satisfaction Score'], errors='coerce')

df['resolution_hours'] = (df['Completion Date'] - df['Call Date']).dt.total_seconds() / 3600
df['sla_met'] = np.where(df['Completion Date'].isna(), np.nan,
                          df['Completion Date'] <= df['Target Date'])

df['call_year']    = df['Call Date'].dt.year
df['call_month']   = df['Call Date'].dt.to_period('M').astype(str)
df['call_weekday'] = df['Call Date'].dt.day_name()
df['call_hour']    = df['Call Date'].dt.hour

### Final data after cleaning

In [19]:
df = df.dropna(subset=['Call Date', 'Subcategory'])
df = df.rename(columns=lambda c: re.sub(r'[^0-9a-zA-Z]+', '_', c).strip('_').lower())
df = df.drop(columns=['time_spent_for_first_line'])

print(f"{raw_rows} -> {len(df)} rows, {df.shape[1]} columns")
df.head()

44250 -> 41927 rows, 31 columns


,call_date,call_number,caller_name,store_number,store_name,region,brief_description_details,category,subcategory,status,priority,completion_date,target_date,supplier,completed,operator_group,operator,contact_channel,first_call_resolution,knowledgebaseavailable,self_service_eligible,automation_suitability,reopened_count,customer_satisfaction_score,handling_minutes,resolution_hours,sla_met,call_year,call_month,call_weekday,call_hour
0,2024-11-06 16:44:00,C2411-22359,1091STUK OSWESTRY Store Employee,1091,OSWESTRY,Scotland & NI,store wifi dropping,UK - Store Network,Wifi Connection,Closed,P3,2024-11-26 16:49:00,2024-11-08 16:44:00,UK - Vertex,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,No,No,Low,0,3.0,28.416667,480.083333,0.0,2024,2024-11,Wednesday,16
1,2026-02-07 07:08:00,C2602-20950,1203STUK KENDAL Store Employee,1203,KENDAL,Scotland & NI,body cam dock fault,UK - Store Security,Bwc Docking Controller,Closed,P3,2026-02-19 16:43:00,2026-02-09 07:08:00,NaN,TRUE,UK - SD IT Service Desk,Sofia Almeida,Phone,No,Yes,No,Low,0,3.0,100.416667,297.583333,0.0,2026,2026-02,Saturday,7
2,2026-05-18 09:58:00,C2605-20993,1854STUK CARLISLE Store Employee,1854,CARLISLE,London & South East,password reset request,UK - Store Access,Password Reset - Intranet,Closed,P4,2026-05-18 10:20:00,2026-05-23 09:58:00,NaN,TRUE,UK - SD IT Service Desk,Daniel Okafor,Email,Yes,Yes,Yes,High,0,4.0,7.633333,0.366667,1.0,2026,2026-05,Monday,9
3,2026-05-12 12:00:00,C2605-20337,1014STUK CARLISLE Store Employee,1014,CARLISLE,London & South East,chip and pin not taking payments,UK - Store Till,Chip & Pin,Closed,P3,2026-05-20 09:54:00,2026-05-14 12:00:00,NaN,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,Yes,No,Low,0,NaN,27.200000,189.900000,0.0,2026,2026-05,Tuesday,12
4,2024-11-25 09:19:00,C2411-20650,1560STUK YEOVIL Store Employee,1560,YEOVIL,London & South East,body cam offline,UK - Store Security,Bwc Camera Offline,Closed,P3,2024-12-04 19:47:00,2024-11-27 09:19:00,NaN,TRUE,UK - SD IT Service Desk,Liam Doyle,Phone,No,No,No,Low,0,5.0,16.716667,226.466667,0.0,2024,2024-11,Monday,9


In [20]:
df.to_csv('cleaned_service_desk.csv', index=False)

In [21]:
%pip install --user sqlalchemy "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.


In [22]:
df = pd.read_csv('cleaned_service_desk.csv', parse_dates=['call_date','completion_date','target_date'])

In [25]:
from sqlalchemy import create_engine, URL, text

url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password="Vishwaja123",
    host="localhost",
    port=5432,
    database="it_servicedesk_db",
)

engine = create_engine(url)

# Test connection
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).fetchone()[0])

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit


In [27]:
df.to_sql('service_desk_tickets', engine, if_exists='replace',
          index=False, chunksize=1000)

-42

In [28]:
pd.read_sql('SELECT COUNT(*) AS rows FROM service_desk_tickets', engine)

,rows
0,41927
